# DentalGPT hierarchical panoramic X-ray pipeline
Run the cells in order. Start with `BASIC`, then compare the two deeper modes on the same image. The orchestrator receives only DentalGPT text observations, never the X-ray.

In [ ]:
# Kaggle setup. Internet must be enabled for the first model download.
%pip install -q -U 'transformers>=4.51.0' accelerate bitsandbytes qwen-vl-utils openai pydantic

In [ ]:
# Imports
import os
import sys
import json
from pathlib import Path

PROJECT_DIR = Path('/kaggle/working/dentalgpt_kaggle')
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

from dentalgpt import DentalGPTRunner
from pipeline import DentalAnalysisPipeline, OpenAIOrchestrator

In [ ]:
# Main configuration
ANALYSIS_MODE = 'BASIC'  # BASIC | DISEASE_HIERARCHY | DISEASE_AND_LOCATION
IMAGE_PATH = '/kaggle/input/YOUR_DATASET/YOUR_IMAGE.jpg'

MODEL_ID = 'Eric3200/DentalGPT-7B-1026'
PROCESSOR_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'
LOAD_IN_4BIT = True
MAX_NEW_TOKENS = 256

USE_OPENAI_ORCHESTRATOR = True
ORCHESTRATOR_MODEL = 'gpt-5.6'  # Change if this model is unavailable to your account.
OUTPUT_DIR = '/kaggle/working/dental_outputs'

In [ ]:
# Read optional tokens from Kaggle Secrets. Add OPENAI_API_KEY when using the orchestrator.
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    try:
        HF_TOKEN = secrets.get_secret('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
    if USE_OPENAI_ORCHESTRATOR:
        os.environ['OPENAI_API_KEY'] = secrets.get_secret('OPENAI_API_KEY')
except ImportError:
    HF_TOKEN = os.getenv('HF_TOKEN')

print('HF token configured:', bool(HF_TOKEN))
print('OpenAI key configured:', bool(os.getenv('OPENAI_API_KEY')))

In [ ]:
# Download (first run) and load DentalGPT in 4-bit on the Kaggle GPU.
dental_runner = DentalGPTRunner(
    model_id=MODEL_ID,
    processor_id=PROCESSOR_ID,
    load_in_4bit=LOAD_IN_4BIT,
    max_new_tokens=MAX_NEW_TOKENS,
    hf_token=HF_TOKEN,
)
print('DentalGPT loaded.')

In [ ]:
# Build the pipeline. Set USE_OPENAI_ORCHESTRATOR=False to save only raw DentalGPT answers.
orchestrator = (
    OpenAIOrchestrator(model=ORCHESTRATOR_MODEL)
    if USE_OPENAI_ORCHESTRATOR else None
)
pipeline = DentalAnalysisPipeline(dental_runner, orchestrator)

In [ ]:
# Run one image. Start with BASIC because it uses only four DentalGPT calls.
result = pipeline.run(
    image_path=IMAGE_PATH,
    mode=ANALYSIS_MODE,
    output_dir=OUTPUT_DIR,
)
print('Saved:', result['saved_to'])
print('DentalGPT calls:', result['dentalgpt_call_count'])

In [ ]:
# Show the final report and canonical findings.
if result['orchestrated_output']:
    print(result['orchestrated_output']['report'])
    display(result['orchestrated_output']['findings'])
else:
    display(result['observations'])